# Conversion policy guide

This notebook explains the conversion policy of `spacecore`.

It describes:

- which problem the conversion policy solves,
- how context is chosen,
- how conversion to other context happens,
- how dtype is chosen during conversion,
- which flags regulate conversion behavior,
- how backend-specific default dtypes are determined.

## 1. Motivation: why conversion policy is needed

The library is built around the chain

$$
\texttt{BackendOps} \to \texttt{Context} \to \texttt{Space} \to \texttt{LinOp}.
$$

This means that spaces and operators always live under a numerical policy: a backend, a dtype, and a checking policy.

As soon as objects may come from different sources, the library must answer questions such as:

- Which backend should be used?
- Which dtype should be used?
- Should the library keep the native dtype of an existing object, or switch to the target backend default?
- Should backend changes happen silently, or should the user be warned?

Without an explicit conversion policy, these situations become ambiguous.

### Typical example

Suppose you want to create `ProductSpace` passing $n$ `Space` objects `s1`, ..., `sn`

- the global default context is NumPy with `float64`,
- suppose that for certain $k$, space `sk` has context with `JAX` backend
- and you create the `ProductSpace` object `ps` from `s1`, ..., `sn` without passing `ctx=...` explicitly.

In this case, the library must decide which context `ps` should have.
Possible options:
- infer the context from the input spaces `s1`, ..., `sn` (if it is possible - see below; in this case, no backend conversion of the spaces),
- use the global default context, convert input `Space` objects to the global context.

It must also decide what to do with the dtype associated with each `s1`, ..., `sn`.

So the conversion policy exists to make these decisions deterministic and predictable.

## 2. Context Resolution Policy

The context resolution procedure is illustrated in the figure below:

![context_decision_tree.svg](./img/context_decision_tree.svg)

1. An explicit context is a context provided at initialization via the `ctx` parameter. In the case if explicit context specified only with `str`, all the missing context parameters are set to default.
2. If no explicit context is provided, and some input objects carry a `ctx` attribute, the library attempts to infer a context from those input objects. If there is no such, opt "no" arrow.
3. A common context can be inferred only if all input objects that carry a `ctx` attribute use the same backend.
4. The inferred context is constructed from the shared backend and the most general dtype among those input objects. All other parameters of the inferred context are set to their default values.
5. The global default context is the context set via the `spacecore.set_context()`. To see current global default context, run `spacecore.get_context()`.

Once the context is resolved, it is assigned to the object being created.
Any input parameters that carry a `ctx` attribute are converted to the resolved context.
More precisely, they are adapted to the backend of the resolved context. Their native dtype may or may not be preserved, depending on the active dtype policy; see below.

## 3. How object conversion happens

Assume a particular object `foo` needs to be converted to the new context `new_ctx`.
Schematically, the procedure is as follows:

![convert_to_new_ctx.svg](./img/convert_to_new_ctx.svg)

Dtype of the converted object is treated independently.

### 3.1. Dtype Resolution During Conversion

Dtype resolution is a separate concern from context resolution.

Context resolution determines which context will be assigned to the object being created. Once that context is fixed, input objects carrying their own contexts may need to be converted to it. During this conversion, the backend is determined by the resolved context, but the dtype policy may still vary.

For example, if an input object has dtype `float32` and is converted to a context on a different backend, the library must decide whether to

1. preserve the original dtype, or
2. cast the object to the default dtype of the resolved context.

This behavior is governed by the dtype resolution policy.

Conceptually, the policy controls whether conversion should prioritize dtype preservation or dtype unification.

#### `dtype_resolution_policy`

The resolver supports two dtype policies:

- `convert`
- `keep_native`

These regulate how the dtype is chosen when a new context is normalized relative to an existing one.
The default is `keep_native`.

The dtype resolution policy could be overwritten via `spacecore.set_dtype_resolution_policy()`.
To see current dtype resolution policy, run `spacecore.get_dtype_resolution_policy()`.


##### `convert`

Under `convert`, when an object is converted to the resolved context **with** the dtype that new context provides.

##### `keep_native`

Under `keep_native`, the dtype of the object will be converted to **equivalent** dtype of the backend that new context has.

#### 3.1.1. How inferred dtype is chosen from several objects

When context is inferred from several objects:

- the backend is inferred only if all the objects have the same backend; else, **no context** can be inferred,
- if all inferred dtypes agree, that dtype is kept and normalized to the inferred backend,
- if inferred dtypes differ, the inferred dtype is chosen as the most general dtype among the relevant input objects.

Schematically,

$$
\{d_1, \dots, d_k\}
\mapsto
\begin{cases}
d_1, & \text{if all } d_i \text{ are equal},\\
\texttt{join}(d_1, \dots, d_k), & \text{otherwise}.
\end{cases}
$$
Before joining, dtypes are normalized to the inferred backend.

### 3.2. How other context parameters are chosen from several objects

#### 3.2.1. `enable_checks`

When a context is inferred from several source objects, the inferred checks flag is the conjunction

$$
\texttt{enable\_checks} = \bigwedge_i \texttt{ctx}_i.\texttt{enable\_checks}.
$$

So inferred checks remain enabled only if all source contexts have checks enabled.

## 4. `resolution_policy`: what regulates conversion warnings and errors

The resolver also has a context-conversion policy controlling what happens when the native context of an object and the target context are backend-incompatible.
That is, when object is converted to other than its own backend.

The available policies are:

- `warning`
- `error`
- `silent`

This policy is used when conversion is enforced through the context manager.
The resolution policy could be overwritten via `spacecore.set_resolution_policy()`.
To see active resolution policy, run `spacecore.get_resolution_policy()`.


### 4.1. `warning`

If the object has a native context and the target context has a different backend family, conversion is allowed but a warning is issued.

This is the default behavior.


### 4.2. `error`

If the object has a native context and the target context is backend-incompatible, conversion is rejected.

So this mode is useful when you want to forbid accidental backend migration.


### 4.3. `silent`

If the object has a native context and the target context is backend-incompatible, conversion proceeds without warning.

So this mode is useful when you want smooth automatic conversion and already trust your pipeline.

## Summary

When a new object is created, the library first resolves its context:

1. use the explicit context provided via `ctx`, if any;
2. otherwise, infer a context from input objects that carry a `ctx` attribute, if possible;
3. otherwise, use the global default context.

Context inference is possible only when all context-carrying input objects agree on the same backend. The inferred dtype is then chosen as the most general dtype among those objects, while all other context parameters are set to default values.

Once the context is resolved, it is assigned to the new object, and any context-carrying inputs are converted to it. This conversion fixes the backend, while dtype handling is governed separately by the active dtype policy.

# Appendix

## A. Backend-specific default dtypes and `sanitize_dtype`

Each backend implementation defines its own dtype normalization rule through

```python
sanitize_dtype(dtype)
```

This method serves two purposes:

1. normalize the dtype into the backend's native dtype representation,
2. determine the backend default dtype when `dtype=None`.

So every `BackendOps` implementation has its own default dtype policy.


### A.1 NumPy

For `NumpyOps`,

$$
\texttt{sanitize\_dtype(None)} = \texttt{numpy.float64}.
$$

If a dtype is provided, it is normalized by `numpy.dtype(...)`.

So NumPy's default dtype policy in the current implementation is `float64`.


### A.2 JAX

For `JaxOps`, the default dtype depends on the JAX configuration.

- if `jax_enable_x64=True`, the default is `float64`,
- otherwise, the default is `float32`, and the backend warns that this is the active default.

Also, `JaxOps.sanitize_dtype(...)` rejects dtypes that are not valid or not permitted under the current JAX configuration.

So the default dtype is backend-specific and, for JAX, also configuration-dependent.


## B. Default values

- Default `enable_checks` is set to `False`.
- Default dtype is set to `None`.
- Default context has `NumPy` backend. Other parameters are default.
- Default resolution policy is `warning`.
- Default dtype resolution policy in `keep_native`.